# Guwahati LSTM - Stage 4 ML-Ready Data

This notebook trains:
- `CLOUD_Vertical_Visibility`
- `VIS_Visibility_MOR`

It uses only the output of `prepare_guwahati_vertical_visibility_mlready.py` and a chronological split.

In [4]:
from pathlib import Path
import json, random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "scripts" / "prepare_guwahati_vertical_visibility_mlready.py").exists():
    raise FileNotFoundError("Run this notebook from the repository or notebooks directory")
DATA_FILE = PROJECT_ROOT / "07_lstm_ready" / "vertical_visibility_experiment" / "guwahati_mlready.csv"
ARTIFACT_DIR = PROJECT_ROOT / "10_models" / "guwahati_vertical_visibility_experiment"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMNS = ["CLOUD_Vertical_Visibility", "VIS_Visibility_MOR"]
LOOKBACK = 96
HORIZON = 1
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
BATCH_SIZE = 256
EPOCHS = 50
PATIENCE = 8

FEATURE_COLUMNS = [
    "CLOUD_Cloud_Base_1", "CLOUD_Cloud_Base_2", "CLOUD_Cloud_Base_3",
    "CLOUD_OCTA1", "CLOUD_OCTA2", "CLOUD_OCTA3", "CLOUD_OCTA4",
    "PTU_Pressure_QNH", "PTU_Temperature", "PTU_DewPoint", "PTU_Humidity",
    "PW_Rain_Intensity", "PW_Rain_Sum", "VIS_RVR",
    "WIND_Wind_Dir_Sin", "WIND_Wind_Dir_Cos", "WIND_Wind_Speed", "WIND_Wind_Gust",
    "Hour_Sin", "Hour_Cos", "CLOUD_SENSOR_ON", "PTU_SENSOR_ON", "PW_SENSOR_ON",
    "VIS_SENSOR_ON", "WIND_SENSOR_ON", "CLOUD_Cloud_Base_1_IMPUTED",
    "CLOUD_Cloud_Base_2_IMPUTED", "CLOUD_Cloud_Base_3_IMPUTED",
    "CLOUD_Vertical_Visibility_IMPUTED", "CLOUD_LONG_GAP", "CLOUD_VERTICAL_LONG_GAP", "ROW_GAP",
]

assert DATA_FILE.exists(), DATA_FILE
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Data:", DATA_FILE)


PyTorch: 2.5.1
Device: cuda
Data: <repository>/07_lstm_ready/vertical_visibility_experiment/guwahati_mlready.csv


In [5]:
df = pd.read_csv(DATA_FILE, parse_dates=["Datetime"], dayfirst=True)
df = df.sort_values("Datetime").drop_duplicates("Datetime")

required_raw = sorted(set(FEATURE_COLUMNS + TARGET_COLUMNS) - {
    "WIND_Wind_Dir_Sin", "WIND_Wind_Dir_Cos", "ROW_GAP"
})
required_raw = sorted(set(required_raw) | {"WIND_Wind_Dir"})
missing = sorted(set(required_raw) - set(df.columns))
assert not missing, f"Missing raw columns: {missing}"

numeric_columns = sorted((set(FEATURE_COLUMNS + TARGET_COLUMNS) - {
    "WIND_Wind_Dir_Sin", "WIND_Wind_Dir_Cos", "ROW_GAP"
}) & set(df.columns))
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")
df.loc[df["WIND_Wind_Speed"] < 0, "WIND_Wind_Speed"] = np.nan
df.loc[df["WIND_Wind_Gust"] < 0, "WIND_Wind_Gust"] = np.nan
df["WIND_Wind_Dir_Sin"] = np.sin(np.deg2rad(pd.to_numeric(df["WIND_Wind_Dir"], errors="coerce")))
df["WIND_Wind_Dir_Cos"] = np.cos(np.deg2rad(pd.to_numeric(df["WIND_Wind_Dir"], errors="coerce")))
df = df.set_index("Datetime")
df["ROW_GAP"] = df.index.to_series().diff().dt.total_seconds().gt(15).fillna(False)

print("Rows:", len(df), "range:", df.index.min(), df.index.max())
print("Valid cloud labels:", int(df["CLOUD_Vertical_Visibility"].notna().sum()))
print("Valid MOR labels:", int(df["VIS_Visibility_MOR"].notna().sum()))


Rows: 2848084 range: 2024-08-11 00:00:00 2026-01-31 10:08:45
Valid cloud labels: 2848084
Valid MOR labels: 1889019


In [6]:
feature_scaler = StandardScaler()
target_scaler = StandardScaler()

train_end = int(len(df) * TRAIN_RATIO)
feature_frame = df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
feature_medians = feature_frame.iloc[:train_end].median().fillna(0.0)
feature_scaler.fit(feature_frame.iloc[:train_end].fillna(feature_medians))
prepared = pd.DataFrame(feature_scaler.transform(feature_frame.fillna(feature_medians)), index=df.index, columns=FEATURE_COLUMNS).fillna(0).astype("float32")

target_training = df.iloc[:train_end][TARGET_COLUMNS].replace([np.inf, -np.inf], np.nan)
mor_valid = target_training["VIS_Visibility_MOR"].notna()
if "VIS_SENSOR_ON" in df.columns:
    mor_valid &= df.iloc[:train_end]["VIS_SENSOR_ON"].fillna(0).eq(1)
target_training.loc[~mor_valid, "VIS_Visibility_MOR"] = np.nan
target_medians = target_training.median().fillna(0.0)
target_scaler.fit(target_training.fillna(target_medians))

values = prepared.to_numpy(dtype="float32")
targets = df[TARGET_COLUMNS].replace([np.inf, -np.inf], np.nan).to_numpy(dtype="float32")
mor_measured = df["VIS_Visibility_MOR"].notna().to_numpy(dtype=bool)
mor_sensor_on = df["VIS_SENSOR_ON"].fillna(1).eq(1).to_numpy(dtype=bool) if "VIS_SENSOR_ON" in df.columns else mor_measured

cloud_loss_mask = np.isfinite(targets[:, 0])
mor_loss_mask = mor_measured & mor_sensor_on & np.isfinite(targets[:, 1])
loss_mask = np.column_stack([cloud_loss_mask, mor_loss_mask])
gaps = df["ROW_GAP"].to_numpy(dtype=bool)

window_ends, labels, label_masks, timestamps = [], [], [], []
for end in range(LOOKBACK, len(df) - HORIZON + 1):
    horizon = end + HORIZON - 1
    if gaps[end - LOOKBACK + 1:horizon + 1].any():
        continue
    if not loss_mask[horizon].any():
        continue
    label = targets[horizon].copy()
    label_mask = loss_mask[horizon].copy()
    label[~label_mask] = target_medians.to_numpy()[~label_mask]
    window_ends.append(end)
    labels.append(label)
    label_masks.append(label_mask.astype("float32"))
    timestamps.append(df.index[horizon])

window_ends = np.asarray(window_ends, dtype="int64")
y_original = np.asarray(labels, dtype="float32")
mask = np.asarray(label_masks, dtype="float32")
timestamps = pd.to_datetime(timestamps)
y_scaled = target_scaler.transform(y_original).astype("float32")

train_cutoff = timestamps[int(len(timestamps) * TRAIN_RATIO)]
validation_cutoff = timestamps[int(len(timestamps) * (TRAIN_RATIO + VALIDATION_RATIO))]
train = timestamps < train_cutoff
validation = (timestamps >= train_cutoff) & (timestamps < validation_cutoff)
test = timestamps >= validation_cutoff

print("Windows:", len(window_ends), "train/validation/test:", int(train.sum()), int(validation.sum()), int(test.sum()))


Windows: 2839701 train/validation/test: 1987790 425955 425956


c:\Users\iiiaamnooob\miniconda3\envs\weather_pipe\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [7]:
class MultiOutputLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(len(FEATURE_COLUMNS), 64, batch_first=True)
        self.head = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, len(TARGET_COLUMNS)))
    def forward(self, values):
        output, _ = self.lstm(values)
        return self.head(output[:, -1, :])

class WindowDataset(torch.utils.data.Dataset):
    def __len__(self):
        return len(window_ends)
    def __getitem__(self, index):
        end = window_ends[index]
        return torch.from_numpy(values[end - LOOKBACK:end]), torch.from_numpy(y_scaled[index]), torch.from_numpy(mask[index])

dataset = WindowDataset()
def loader(split_mask):
    indices = np.flatnonzero(split_mask).tolist()
    return DataLoader(Subset(dataset, indices), batch_size=BATCH_SIZE, shuffle=False)

train_loader = loader(train)
validation_loader = loader(validation)
model = MultiOutputLSTM().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss(reduction="none")

best_loss = float("inf")
best_state = None
no_improvement = 0
for epoch in range(EPOCHS):
    model.train()
    for inputs, labels, label_mask in train_loader:
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)
        label_mask = label_mask.to(DEVICE)
        optimizer.zero_grad()
        loss = (criterion(model(inputs), labels) * label_mask).sum() / label_mask.sum().clamp_min(1)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    model.eval()
    losses = []
    with torch.no_grad():
        for inputs, labels, label_mask in validation_loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)
            label_mask = label_mask.to(DEVICE)
            loss = (criterion(model(inputs), labels) * label_mask).sum() / label_mask.sum().clamp_min(1)
            losses.append(loss.item())

    validation_loss = float(np.mean(losses)) if losses else float("inf")
    print(f"Epoch {epoch + 1:03d}/{EPOCHS} - val_loss: {validation_loss:.6f}")
    if validation_loss < best_loss:
        best_loss = validation_loss
        no_improvement = 0
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    else:
        no_improvement += 1
        if no_improvement >= PATIENCE:
            print("Early stopping.")
            break

if best_state is not None:
    model.load_state_dict(best_state)
torch.save({"model_state_dict": model.state_dict(), "features": FEATURE_COLUMNS, "targets": TARGET_COLUMNS}, ARTIFACT_DIR / "best_guwahati_cloud_lstm_stage4.pt")


Epoch 001/50 - val_loss: 0.170311
Epoch 002/50 - val_loss: 0.114262
Epoch 003/50 - val_loss: 0.122494
Epoch 004/50 - val_loss: 0.097736
Epoch 005/50 - val_loss: 0.091687
Epoch 006/50 - val_loss: 0.091607
Epoch 007/50 - val_loss: 0.091083
Epoch 008/50 - val_loss: 0.079298
Epoch 009/50 - val_loss: 0.084433
Epoch 010/50 - val_loss: 0.081933
Epoch 011/50 - val_loss: 0.079096
Epoch 012/50 - val_loss: 0.075283
Epoch 013/50 - val_loss: 0.080158
Epoch 014/50 - val_loss: 0.077885
Epoch 015/50 - val_loss: 0.077259
Epoch 016/50 - val_loss: 0.082649
Epoch 017/50 - val_loss: 0.082010
Epoch 018/50 - val_loss: 0.087105
Epoch 019/50 - val_loss: 0.087364
Epoch 020/50 - val_loss: 0.084106
Early stopping.


In [8]:
test_loader = loader(test)
model.eval()
prediction_batches = []
with torch.no_grad():
    for inputs, _, _ in test_loader:
        prediction_batches.append(model(inputs.to(DEVICE)).cpu().numpy())
predictions = target_scaler.inverse_transform(np.concatenate(prediction_batches, axis=0))

window_timestamps_test = timestamps[test]
timestamp_to_row = pd.Series(np.arange(len(df)), index=df.index)
test_row_positions = np.array([int(timestamp_to_row.loc[t]) for t in window_timestamps_test])
actual = y_original[test].copy()
actual[:, 0] = df["CLOUD_Vertical_Visibility"].to_numpy(dtype="float32")[test_row_positions]
actual[:, 1] = df["VIS_Visibility_MOR"].to_numpy(dtype="float32")[test_row_positions]

cloud_test_valid = np.isfinite(actual[:, 0])
mor_test_valid = np.isfinite(actual[:, 1]) & mor_sensor_on[test_row_positions]
test_valid = np.column_stack([cloud_test_valid, mor_test_valid])

rows = []
for index, target in enumerate(TARGET_COLUMNS):
    actual_target = actual[test_valid[:, index], index]
    predicted_target = predictions[test_valid[:, index], index]
    rows.append({"target": target, "measured_samples": len(actual_target), "MAE": mean_absolute_error(actual_target, predicted_target), "RMSE": mean_squared_error(actual_target, predicted_target) ** 0.5, "R2": r2_score(actual_target, predicted_target) if len(actual_target) > 1 else np.nan})
metrics = pd.DataFrame(rows)
metrics.to_csv(ARTIFACT_DIR / "test_metrics.csv", index=False)
display(metrics)


,target,measured_samples,MAE,RMSE,R2
0,CLOUD_Vertical_Visibility,425956,42.067303,237.942056,0.976313
1,VIS_Visibility_MOR,377760,1384.716431,1959.342364,0.557270
